# 📈 Notebook 4: Full Evaluation & Comparison

This notebook loads all previously computed evaluation results and produces comprehensive comparisons across:
- Content-Based (TF-IDF vs BoW)
- Collaborative Filtering (all methods)
- Combined leaderboard

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})


## 1. Load All Results

In [ ]:
results_cb = pd.read_csv('results_content_based.csv', index_col=0)
results_cf = pd.read_csv('results_collaborative_filtering.csv', index_col=0)

# Add approach label
results_cb['Approach'] = 'Content-Based'
results_cf['Approach'] = 'Collaborative Filtering'

all_results = pd.concat([results_cb, results_cf])
print("All models:")
print(all_results[['Approach'] + [c for c in all_results.columns if c != 'Approach']].to_string())


## 2. Metric Definitions

In [ ]:
metric_info = {
    'HitRate@10'   : ('Higher ↑', 'At least one relevant item in top-10'),
    'Precision@10' : ('Higher ↑', 'Relevant items / 10 in top-10'),
    'Recall@10'    : ('Higher ↑', 'Relevant items found / total relevant'),
    'NDCG@10'      : ('Higher ↑', 'Rank-aware relevance score'),
    'RMSE'         : ('Lower ↓',  'Root mean squared prediction error'),
    'MAE'          : ('Lower ↓',  'Mean absolute prediction error'),
    'Diversity'    : ('Higher ↑', 'Avg pairwise distance in recommendations'),
    'Novelty'      : ('Higher ↑', '-log2(popularity) of recommended items'),
    'Coverage'     : ('Higher ↑', 'Fraction of catalogue recommended'),
    'Serendipity'  : ('Higher ↑', 'Unexpected but relevant recommendations'),
}
print("\nMetric reference:")
for m, (direction, desc) in metric_info.items():
    print(f"  {m:15s}  {direction}   {desc}")


## 3. Per-Metric Bar Charts (All Models)

In [ ]:
metrics = [c for c in all_results.columns if c != 'Approach']
n = len(all_results)
cb_n  = len(results_cb)
cf_n  = len(results_cf)
colors_cb = ['#2196F3', '#FF9800']                              # TF-IDF, BoW
colors_cf = cm.Set2(np.linspace(0, 1, cf_n))

all_colors = list(colors_cb) + [list(c) for c in colors_cf]

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
axes = axes.flatten()

for i, metric in enumerate(metrics):
    ax = axes[i]
    vals = all_results[metric]
    bars = ax.bar(range(n), vals, color=all_colors, edgecolor='white', width=0.7)
    direction = metric_info.get(metric, ('Higher ↑', ''))[0]
    ax.set_title(f'{metric}  ({direction})', fontweight='bold', fontsize=9)
    ax.set_xticks(range(n))
    ax.set_xticklabels(all_results.index, rotation=55, ha='right', fontsize=7)
    ax.set_ylim(0, max(vals.max() * 1.35, 1e-6))
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + vals.max()*0.02,
                f'{h:.3f}', ha='center', va='bottom', fontsize=6, rotation=0)

plt.suptitle('All Models — All Metrics', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()


## 4. Radar / Spider Chart

In [ ]:
from matplotlib.patches import FancyArrowPatch
from matplotlib.path import Path

# Normalise metrics 0-1 (flip RMSE/MAE)
norm_df = all_results[metrics].copy()
for m in metrics:
    col = norm_df[m].replace([np.inf, -np.inf], np.nan).dropna()
    mn, mx = col.min(), col.max()
    if mx - mn < 1e-9:
        norm_df[m] = 0.5
    elif m in ('RMSE', 'MAE'):     # lower is better → invert
        norm_df[m] = 1 - (norm_df[m] - mn) / (mx - mn)
    else:
        norm_df[m] = (norm_df[m] - mn) / (mx - mn)
norm_df = norm_df.fillna(0)

angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw=dict(polar=True))

for ax, (grp_name, grp_idx) in zip(axes, [('Content-Based', results_cb.index),
                                            ('Collaborative Filtering', results_cf.index)]):
    ax.set_title(grp_name, size=12, fontweight='bold', pad=15)
    palette = plt.cm.Set1(np.linspace(0, 1, len(grp_idx)))
    for color, model in zip(palette, grp_idx):
        vals = norm_df.loc[model].tolist()
        vals += vals[:1]
        ax.plot(angles, vals, color=color, linewidth=2, label=model)
        ax.fill(angles, vals, color=color, alpha=0.1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, size=8)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)

plt.suptitle('Radar Chart — Normalised Metrics', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()


## 5. Heatmap Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Content-Based
sns.heatmap(results_cb[metrics].astype(float), annot=True, fmt='.4f',
            cmap='YlGnBu', linewidths=0.5, ax=axes[0],
            cbar_kws={'shrink': 0.6})
axes[0].set_title('Content-Based Models', fontweight='bold')
axes[0].set_xticklabels(metrics, rotation=45, ha='right')

# Collaborative Filtering
sns.heatmap(results_cf[metrics].astype(float), annot=True, fmt='.4f',
            cmap='YlOrRd', linewidths=0.5, ax=axes[1],
            cbar_kws={'shrink': 0.6})
axes[1].set_title('Collaborative Filtering Models', fontweight='bold')
axes[1].set_xticklabels(metrics, rotation=45, ha='right')

plt.tight_layout(); plt.show()


## 6. TF-IDF vs BoW (Content-Based Deep Dive)

In [ ]:
cb = results_cb[metrics].astype(float)
x  = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
bars1 = ax.bar(x - width/2, cb.loc['TF-IDF'], width, label='TF-IDF', color='#2196F3', edgecolor='white')
bars2 = ax.bar(x + width/2, cb.loc['BoW'],    width, label='BoW',    color='#FF9800', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(metrics, rotation=30, ha='right')
ax.set_title('Content-Based: TF-IDF vs BoW', fontweight='bold')
ax.legend()

for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=7, rotation=90)

plt.tight_layout(); plt.show()

# Difference table
diff = cb.loc['TF-IDF'] - cb.loc['BoW']
print("\nTF-IDF minus BoW (positive = TF-IDF better for ↑ metrics):")
print(diff.round(4).to_string())


## 7. Best Model Per Metric

In [ ]:
best = {}
for metric in metrics:
    col = all_results[metric].dropna()
    if metric in ('RMSE', 'MAE'):
        best[metric] = col.idxmin()
    else:
        best[metric] = col.idxmax()

print("Best model per metric:")
print("-" * 40)
for m, model in best.items():
    val = all_results.loc[model, m]
    direction = metric_info[m][0]
    print(f"  {m:15s}  →  {model:30s}  ({val:.4f})  {direction}")


## 8. Overall Leaderboard (Composite Score)

In [ ]:
# Normalise and compute composite score
composite = norm_df.mean(axis=1).sort_values(ascending=False)

print("\n=== OVERALL LEADERBOARD (normalised composite score) ===")
print("-" * 50)
for rank, (model, score) in enumerate(composite.items(), 1):
    approach = all_results.loc[model, 'Approach']
    print(f"  #{rank:2d}  {model:30s}  {score:.4f}  [{approach}]")

plt.figure(figsize=(10, 5))
colors_bar = ['#2196F3' if all_results.loc[m,'Approach']=='Content-Based' else '#E91E63'
              for m in composite.index]
plt.barh(composite.index[::-1], composite.values[::-1], color=colors_bar[::-1], edgecolor='white')
plt.xlabel('Composite Score (normalised avg across all metrics)')
plt.title('Model Leaderboard', fontweight='bold')
from matplotlib.patches import Patch
legend_els = [Patch(facecolor='#2196F3', label='Content-Based'),
              Patch(facecolor='#E91E63', label='Collaborative Filtering')]
plt.legend(handles=legend_els, loc='lower right')
plt.tight_layout(); plt.show()


## 9. Key Takeaways

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║               EVALUATION SUMMARY & INSIGHTS                 ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  Content-Based Filtering                                     ║
║  ─────────────────────────────────────────────────────────  ║
║  • TF-IDF vs BoW differ mainly in Diversity & Coverage.     ║
║  • TF-IDF captures richer term importance weighting.        ║
║  • BoW is simpler but can be equally effective on short     ║
║    genre-tag documents.                                      ║
║                                                              ║
║  Collaborative Filtering                                     ║
║  ─────────────────────────────────────────────────────────  ║
║  • KNN (user/item) offers strong personalisation.           ║
║  • Pearson correlation is robust to rating scale biases.    ║
║  • SVD / MF achieve best RMSE by learning latent factors.   ║
║  • Item-based methods tend to be more stable (sparse data). ║
║  • TF-IDF item sims generally outperform BoW item sims      ║
║    in recall and NDCG.                                       ║
║                                                              ║
║  General                                                     ║
║  ─────────────────────────────────────────────────────────  ║
║  • CF has higher Coverage & Novelty; CB has higher          ║
║    Serendipity.                                              ║
║  • Combining both (hybrid) would likely outperform either.  ║
╚══════════════════════════════════════════════════════════════╝
""")
